In [102]:
import re
from util_order import *
import pandas as pd
import random
from cal_sel import *
import os

dataA = pd.read_csv("./datalake/ground_truth/nba.csv")
dataA = dataA.sample(97)

dataB = pd.read_csv("./datalake/ground_truth/team.csv")
dataB = dataB.sample(30)

from llm import *
# Your API key
#OPENAI_KEY = 'sk-X5lP13FSF9jrOFMx7d3dC460E9B94dB792A25c0cF140EeBa'
OPENAI_KEY = 'sk-2xcZcn3QLTX4LHJEB6A80aEc43F04dB18c44Bc70Ac3208F7'

# Initialize the OpenAI API
init_chatgpt(OPENAI_KEY)

In [ ]:
sql_query = """
SELECT A.name, A.team , B.team , B.founded_year FROM A
JOIN B ON A.team = B.team
WHERE A.age > 30 AND B.founded_year > 1940 AND A.team = Los Angeles Lakers
"""

select_clause = re.findall(r"SELECT(.*?)FROM", sql_query, re.DOTALL)[0].strip()

from_match = re.search(r"FROM(.*?)(WHERE|$)", sql_query, re.DOTALL)
from_clause = from_match.group(1).strip() if from_match else ''

where_match = re.search(r"WHERE(.*)", sql_query, re.DOTALL)
where_clause = where_match.group(1).strip() if where_match else ''

join_match = re.search(r"JOIN(.*?)ON", from_clause, re.DOTALL)
join_clause = join_match.group(1).strip() if join_match else ''

on_match = re.search(r"ON(.*)", from_clause, re.DOTALL)
on_clause = on_match.group(1).strip() if on_match else ''

print("select_clause:", select_clause)
print("from_clause:", from_clause)
print("where_clause:", where_clause)
print("join_clause:", join_clause)
print("on_clause:", on_clause)


In [ ]:
##############################################
#A_select_attributes, B_select_attributes = get_select_attributes(select_clause)
##############################################
def get_select_attributes(select_clause):
    sql_attributes = select_clause.split(",")
    A_select_attributes = []
    B_select_attributes = []
    for attribute in sql_attributes:
        if "A." in attribute:
            A_select_attributes.append(attribute.split(".")[1].strip())
        elif "B." in attribute:
            B_select_attributes.append(attribute.split(".")[1].strip())
    return A_select_attributes, B_select_attributes

A_select_attributes, B_select_attributes = get_select_attributes(select_clause)
print("A_select_attributes:", A_select_attributes)
print("B_select_attributes:", B_select_attributes)

In [ ]:
########################################
#A_filter_conditions, B_filter_conditions = get_filter_conditions(where_clause)
########################################
def get_filter_conditions(where_clause):
    conditions = where_clause.split("AND")
    A_filter_conditions = []
    B_filter_conditions = []
    for condition in conditions:
        if "A." in condition:
            A_filter_conditions.append(condition.strip())
        elif "B." in condition:
            B_filter_conditions.append(condition.strip())
    return A_filter_conditions, B_filter_conditions
A_where_filter_conditions, B_where_filter_conditions = get_filter_conditions(where_clause)
print("A_where_filter_conditions:", A_where_filter_conditions)
print("B_where_filter_conditions:", B_where_filter_conditions)
where_A_clause = " AND ".join(A_where_filter_conditions)
where_B_clause = " AND ".join(B_where_filter_conditions)
print("where_A_clause:", where_A_clause)
print("where_B_clause:", where_B_clause)


In [ ]:
############################################
#A_join_attributes, B_join_attributes = get_join_attributes(on_clause)
############################################
def get_join_attributes(on_clause):
    attributes = on_clause.split("=")
    A_join_attributes = attributes[0].strip().split(".")[1]
    B_join_attributes = attributes[1].strip().split(".")[1]
    return A_join_attributes, B_join_attributes
A_join_attributes, B_join_attributes = get_join_attributes(on_clause)
print("A_join_attributes:", A_join_attributes)
print("B_join_attributes:", B_join_attributes)
join_filter_conditions = on_clause
print("join_filter_conditions:", join_filter_conditions)

In [107]:
############################################
#Emin = cal_Emin(ordered_filters)
############################################
def cal_Emin(ordered_filters):
    Emin = 0
    sel = 1
    for first_table_filter_data in ordered_filters:
        Emin += sel * first_table_filter_data["cost"]
        sel *= (1 - first_table_filter_data["selectivity"])
    return Emin

############################################
#selectivity_join_A = cal_selectivity_join(A_join_attributes,B_join_attributs, dataA, dataB)
############################################
def cal_selectivity_join(A_join_attributes, B_join_attributes, dataA, dataB):
    join_data = pd.merge(dataA, dataB, how='inner', left_on=A_join_attributes, right_on=B_join_attributes)
    selectivity_join = len(join_data) / len(dataA)
    return selectivity_join



In [ ]:
selectivity_join_A = cal_selectivity_join(A_join_attributes,B_join_attributes, dataA, dataB)
selectivity_join_B = cal_selectivity_join(B_join_attributes,A_join_attributes, dataB, dataA)

print("selectivity_join_A: ",selectivity_join_A)
print("selectivity_join_B: ",selectivity_join_B)

In [109]:
from dataclasses import dataclass

@dataclass
class TableInfo:
    name: str
    file_list: list
    file_path: str
    data: dict
    selectivity_join: float
    where_filter_conditions: list
    where_clause: str
    join_attributes: str
    select_attributes: list
    

In [110]:
file_list_A = os.listdir("./datalake/nbadata")
file_path_A = "./datalake/nbadata/"
file_list_B = os.listdir("./datalake/team")
file_path_B = "./datalake/team/"
table_info_A = TableInfo(
    name='A',
    file_list=file_list_A,
    file_path=file_path_A,
    data=dataA,
    selectivity_join=selectivity_join_A,
    where_filter_conditions=A_where_filter_conditions,
    where_clause=where_A_clause,
    join_attributes=A_join_attributes,
    select_attributes=A_select_attributes
)
table_info_B = TableInfo(
    name='B',
    file_list=file_list_B,
    file_path=file_path_B,
    data=dataB,
    selectivity_join=selectivity_join_B,
    where_filter_conditions=B_where_filter_conditions,
    where_clause=where_B_clause,
    join_attributes=B_join_attributes,
    select_attributes=B_select_attributes
)


In [111]:
############################################
#Eall = cal_Eall(file_list_A, file_list_B,file_path_A, file_path_B, dataA,dataB, selectivity_join_A, selectivity_join_B)
############################################
import os
import time

def cal_Eall(table_info_A, table_info_B):
    file_list_A = table_info_A.file_list
    file_path_A = table_info_A.file_path
    dataA = table_info_A.data
    selectivity_join_A = table_info_A.selectivity_join
    A_where_filter_conditions = table_info_A.where_filter_conditions
    where_A_clause = table_info_A.where_clause
    A_join_attributes = table_info_A.join_attributes

    file_list_B = table_info_B.file_list
    file_path_B = table_info_B.file_path
    dataB = table_info_B.data
    selectivity_join_B = table_info_B.selectivity_join
    B_where_filter_conditions = table_info_B.where_filter_conditions
    where_B_clause = table_info_B.where_clause
    B_join_attributes = table_info_B.join_attributes



    first_table_filter_data = {}
    start_time = time.time()
    
    Emin = 0
    cost_join_attribute = 0
    E_all = 0

    for onefile in file_list_A:
        input_text = ""
        file_path = os.path.join(file_path_A, onefile)
        #print("file:", file_path)
        with open(file_path, "r") as file:
            input_text = file.read()
        
        cur_file_attributes = parse_input(input_text)
    
        #print("filter conditions:", A_where_filter_conditions)
        for filter_condition in A_where_filter_conditions:
            table, cur_filter_condition = filter_condition.split(".")
            attribute, operator, value = cur_filter_condition.split(maxsplit=2)
            #print(f"attribute: {attribute}, operator: {operator}, value: {value}")
            selectivity = cal_sel(dataA, {"name": cur_filter_condition})
            if attribute in cur_file_attributes:
                try:
                    cur_sentence = cur_file_attributes[attribute]['key_sentences']
                    cur_cost = cur_file_attributes[attribute]['cost']
                except KeyError:
                    cur_sentence = ""
                    cur_cost = 0
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": cur_cost,
                    's_value': calculate_s(selectivity, cur_sentence),
                    "key_sentences": cur_sentence
                }
            else:
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": 0,
                    's_value': 0,
                    "key_sentences": []
                }
        
        #print("first_table_filter_data:", first_table_filter_data)
        try:
            sorted_filters = handle_sql(where_A_clause, first_table_filter_data)
        except Exception as e:
            #print("Error processing file:", file_path, "Error:", e)
            continue
        # #print("Sorted Filters:")
        # for filter_cond, s_value in sorted_filters:
        #     print(f"{filter_cond}: S = {s_value:.4f}")
        
        ordered_filters = []
        for filter_cond, s_value in sorted_filters:
            order_filter = first_table_filter_data[filter_cond]
            ordered_filters.append(order_filter)
        #print("ordered_filters:", ordered_filters)
    
        Emin += cal_Emin(ordered_filters)
        #print("Emin:", Emin)
    
        try:
            cur_cost_join_attribute = cur_file_attributes[A_join_attributes]['cost']
        except KeyError:
            cur_cost_join_attribute = 0
        cost_join_attribute += cur_cost_join_attribute
    
    for onefile in file_list_B:
        input_text = ""
        file_path = os.path.join(file_path_B, onefile)
        with open(file_path, "r") as file:
            input_text = file.read()
        
        cur_file_attributes = parse_input(input_text)
    
        for filter_condition in B_where_filter_conditions:
            table, cur_filter_condition = filter_condition.split(".")
            attribute, operator, value = cur_filter_condition.split(maxsplit=2)
            selectivity = cal_sel(dataB, {"name": cur_filter_condition})
            if attribute in cur_file_attributes:
                try:
                    cur_sentence = cur_file_attributes[attribute]['key_sentences']
                    cur_cost = cur_file_attributes[attribute]['cost']
                except KeyError:
                    cur_sentence = ""
                    cur_cost = 0
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": cur_cost,
                    's_value': calculate_s(selectivity, cur_sentence),
                    "key_sentences": cur_sentence
                }
            else:
                first_table_filter_data[filter_condition] = {
                    "name": filter_condition,
                    "table": table,
                    "selectivity": selectivity,
                    "cost": 0,
                    's_value': 0,
                    "key_sentences": []
                }
        if B_join_attributes in cur_file_attributes:
            try:
                cur_cost_join_attribute = cur_file_attributes[B_join_attributes]['cost']
                cur_sentence = cur_file_attributes[B_join_attributes]['key_sentences']
            except KeyError:
                cur_cost_join_attribute = 0
                cur_sentence = ""
            first_table_filter_data["join_filter_condition"] = {
                "name": "join_filter_condition",
                "table": "B",
                "selectivity": selectivity_join_B,
                "cost": cur_cost_join_attribute,
                's_value': calculate_s(selectivity_join_B, cur_sentence),
                "key_sentences": cur_sentence
            }

        try:
            sorted_filters = handle_sql(where_B_clause, first_table_filter_data)
        except Exception as e:
            print("Error processing file:", file_path, "Error:", e)
            continue

        
        ordered_filters = []
        for filter_cond, s_value in sorted_filters:
            order_filter = first_table_filter_data[filter_cond]
            ordered_filters.append(order_filter)
    
        Emin += cal_Emin(ordered_filters)
       
    E_all = Emin + cost_join_attribute*selectivity_join_A
    return E_all

In [112]:
def get_2_table_order(table_info_A, table_info_B):
    where_A_clause = table_info_A.where_clause
    where_B_clause = table_info_B.where_clause
    test_where_B_clause = where_B_clause +  " AND " + "join_filter_condition"
    E1 = cal_Eall(table_info_A, table_info_B)
    print("E1:", E1)
    test_where_A_clause = where_A_clause +  " AND " + "join_filter_condition"
    E2 = cal_Eall(table_info_B, table_info_A)
    print("E2:", E2)
    if E1 < E2:
        return table_info_A, table_info_B
    else:
        return table_info_B, table_info_A

In [ ]:

first_table_info,second_table_info = get_2_table_order(table_info_A, table_info_B)
first_table_path = first_table_info.file_path
second_table_path = second_table_info.file_path
print("first_table_infor:", first_table_path)
print("second_table_infor:", second_table_path)

In [114]:
def rewrite_where_clause(where_clause):
    tmp_filter_cond = extract_filter_condition(where_clause)
    filter_conds = []
    for filter_cond in tmp_filter_cond:
        table, sub_filter_condition = filter_cond.split(".", 1)
        filter_conds.append(sub_filter_condition)
    where_clause = "WHERE " + " AND ".join(filter_conds)
    return where_clause

def rewrite_where_filter_conditions(where_filter_conditions):
    filter_conds = []
    for filter_cond in where_filter_conditions:
        table, sub_filter_condition = filter_cond.split(".", 1)
        filter_conds.append(sub_filter_condition)
    return filter_conds

In [ ]:
first_table_filter_data = {}
all_total_token = 0
all_actual_token = 0
uni_dir = "./result/exp5/"
if not os.path.exists(uni_dir):
    os.makedirs(uni_dir)
first_table_output = pd.DataFrame()

import os
import time
file_list = os.listdir(first_table_path)

infor = uni_dir + "infor.txt"
start_tie = time.time()
initial_where_clause = where_clause

for onefile in file_list:
    input_text = ""
    file_path = first_table_path + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    where_clause = first_table_info.where_clause
    where_clause = rewrite_where_clause(where_clause)
    print("where_clause: ",where_clause)

    filter_conditions = first_table_info.where_filter_conditions
    filter_cond = rewrite_where_filter_conditions(filter_conditions)
    print("filter_cond: ",filter_cond)

    data = first_table_info.data

    attributes = parse_input(input_text)

    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    for filter_condtion in filter_cond:
        print("filter_condtion: ",filter_condtion)
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        try:
            key_sentences = attributes[atrribute]['key_sentences']
            cost = attributes[atrribute]['cost']
        except KeyError:
            key_sentences = ""
            cost = 0
        if atrribute in attributes:
            first_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "table": first_table_info.name,
                "selectivity": selectivity,
                "cost": cost,
                's_value': calculate_s(selectivity, key_sentences),
                "key_sentences": key_sentences
            }
        else:
            first_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }

    print("first_table_filter_data: ",first_table_filter_data)
    print("where_clause: ",where_clause)
    try:
        sort_where_clause = where_clause.replace("WHERE","")
        sorted_filters = handle_sql(sort_where_clause, first_table_filter_data)
    except:
        print("the file is error: ",file_path)
        continue
    print("Sorted Filters:")
    for filter_cond, s_value in sorted_filters:
        print(f"{filter_cond}: S = {s_value:.4f}")

    print("\n")

    actual_token = 0


    skipthefile = 0
    sql_copy = where_clause
    #remian_attributes is a list of attributes that have not been extracted
    select_attributes = first_table_info.select_attributes
    remaining_attributes = select_attributes.copy()

    remaining_attributes = list(set(remaining_attributes))
    first_table_curdata = {}
    first_table_sorted_filter_cond = []
    first_table_mapfilter_cond = {}
    first_table_mapfilter_erro = {}

    for filter in sorted_filters:
        first_table_sorted_filter_cond.append(filter[0])
    for filter in first_table_sorted_filter_cond:
        first_table_mapfilter_cond[filter] = 0
        first_table_mapfilter_erro[filter] = 0

    remaining_first_table_sorted_filter_cond = first_table_sorted_filter_cond.copy()

    while remaining_first_table_sorted_filter_cond:

        filter_cond_tochange = []
        filter_cond = remaining_first_table_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        first_table_mapfilter_cond[filter_cond] += 1
        if first_table_mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remaining_first_table_sorted_filter_cond:
                remaining_first_table_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif first_table_mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remaining_first_table_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            print("answer: \n", answer)
            #attributes_cur_all = answer.split("$$")[1]
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                if 'first_table_output NOT IN REQUIRED FORMAT' in answer and first_table_mapfilter_erro[filter_cond] >= 1:
                    if filter_cond in remaining_first_table_sorted_filter_cond:
                        remaining_first_table_sorted_filter_cond.remove(filter_cond)
                first_table_mapfilter_cond[filter_cond] -= 1
                first_table_mapfilter_erro[filter_cond] += 1
                tochange_filter = [filter_cond,'false']
                actual_token -= len(key_sentences.split())+100
                filter_cond_tochange.append(tochange_filter)
                
                for ask_filter_cond,bool_value in filter_cond_tochange:
                        sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
                sql_copy2 = sql_copy
                print("sql_copy: ",sql_copy)
                bool_value_set_true = calculate_bool_value_true(sql_copy)
                bool_value_set_false = calculate_bool_value_false(sql_copy2)
                print("bool_value_set_true: ",bool_value_set_true)
                print("bool_value_set_false: ",bool_value_set_false)
                if bool_value_set_true == False:
                    skipthefile = 1
                    break
                if bool_value_set_false == True:
                    skipthefile = 0
                    break
                else:
                    skipthefile = 1
                continue

            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    first_table_curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        filter_cond_tochange.append(tochange_filter)
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remaining_first_table_sorted_filter_cond:
                            remaining_first_table_sorted_filter_cond.remove(filter_cond_cur)
                        print("remaining_first_table_sorted_filter_cond: ",remaining_first_table_sorted_filter_cond)
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)

        if bool_value_set_true == False:
            skipthefile = 1
            break

        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")
        #remaining_first_table_sorted_filter_cond.remove(filter_cond)

    if skipthefile == 0:
        
        key_sentences = ""
        first_table_mapattr = {}
        for attr in select_attributes:
            first_table_mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            attr = remaining_attributes[0]
            first_table_mapattr[attr] += 1
            if first_table_mapattr[attr] >= 2:
                remaining_attributes.remove(attr)
                first_table_curdata[attr] = "NAN"
                continue
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+75
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        first_table_curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                remaining_attributes.remove(dataattr)
                except:
                    print("error extract attributes")
                    actual_token -= len(key_sentences.split())+75
                    print("answer: ", answer) 
       
             
        if len(first_table_curdata) != 0:
            first_table_curdata['file'] = onefile
            first_table_output = first_table_output.append(first_table_curdata,ignore_index=True)
            data = data.append(first_table_curdata,ignore_index=True)
            first_table_output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)

with open(infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

first_table_output


In [ ]:
import numpy as np
first_table_output = first_table_output.replace('NAN', np.nan)

first_table_output = first_table_output.dropna(axis=1, how='all')
first_table_output

In [ ]:
join_attr = first_table_info.join_attributes
print("join_attr: ",join_attr)
join_attributes = first_table_output[join_attr].values
join_attributes = list(set(join_attributes))
print("join_attributes: ",join_attributes)

join_attributes = [str(i) for i in join_attributes]
#join_attributes = ['q','a']
join_attributes = ",".join(join_attributes)
join_filter_conditions = join_attr + " IN " + str(join_attributes)
print("join_attributes: ",join_attributes)
print("join_filter_conditions: ",join_filter_conditions)


In [ ]:
second_table_filter_data = {}
uni_dir = "./result/exp5/"
if not os.path.exists(uni_dir):
    os.makedirs(uni_dir)
second_table_output = pd.DataFrame()

import os
import time
second_file_list = os.listdir(second_table_path)

second_infor = uni_dir + "second_infor.txt"
start_tie = time.time()
initial_where_clause = where_clause

for onefile in second_file_list:
    input_text = ""
    file_path = second_table_path + onefile
    print("file: ",file_path)
    with open(file_path, "r") as file:
        input_text = file.read()

    where_clause = second_table_info.where_clause
    where_clause = rewrite_where_clause(where_clause)
    print("where_clause: ",where_clause)

    filter_conditions = second_table_info.where_filter_conditions
    filter_cond = rewrite_where_filter_conditions(filter_conditions)
    print("filter_cond: ",filter_cond)

    data = second_table_info.data

    attributes = parse_input(input_text)

    total_token = 0
    total_token += len(input_text.split()) + 100
    
    print("total_token: ",total_token)
    all_total_token += total_token
    print("all_total_token: ",all_total_token)

    for filter_condtion in filter_cond:
        print("filter_condtion: ",filter_condtion)
        atrribute,operator,value = filter_condtion.split(maxsplit=2)
        print(f"atrribute: {atrribute}, operator: {operator}, value: {value}")
        selectivity = cal_sel(data, {"name": filter_condtion})
        try:
            key_sentences = attributes[atrribute]['key_sentences']
            cost = attributes[atrribute]['cost']
        except KeyError:
            key_sentences = ""
            cost = 0
        if atrribute in attributes:
            second_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "table": second_table_info.name,
                "selectivity": selectivity,
                "cost": cost,
                's_value': calculate_s(selectivity, key_sentences),
                "key_sentences": key_sentences
            }
        else:
            second_table_filter_data[filter_condtion] = {
                "name": filter_condtion,
                "selectivity": selectivity,
                's_value': 0,
                "key_sentences": []
            }
    #print("second_table_filter_data: ",second_table_filter_data)
    try:
            key_sentences = attributes[join_attr]['key_sentences']
            cost = attributes[join_attr]['cost']
    except KeyError:
        key_sentences = ""
        cost = 0
    if atrribute in attributes:
        second_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "table": second_table_info.name,
            "selectivity": second_table_info.selectivity_join,
            "cost": cost,
            's_value': calculate_s(selectivity, key_sentences),
            "key_sentences": key_sentences
        }
    else:
        second_table_filter_data[join_filter_conditions] = {
            "name": join_filter_conditions,
            "selectivity": second_table_info.selectivity_join,
            's_value': 0,
            "key_sentences": []
        }


    print("second_table_filter_data: ",second_table_filter_data)
    print("where_clause: ",where_clause)
    try:
        where_clause += " AND " + join_filter_conditions
        sort_where_clause = where_clause.replace("WHERE","")
        print("sort_where_clause: ",sort_where_clause)
        sorted_filters = handle_sql(sort_where_clause, second_table_filter_data)
    except:
        print("the file is error: ",file_path)
        continue
    print("Sorted Filters:")
    for filter_cond, s_value in sorted_filters:
        print(f"{filter_cond}: S = {s_value:.4f}")
    #continue
    #break
    print("\n")

    actual_token = 0

    skipthefile = 0
    sql_copy = where_clause
    select_attributes = second_table_info.select_attributes
    remaining_attributes = select_attributes.copy()
    remaining_attributes.append(join_attr)

    remaining_attributes = list(set(remaining_attributes))
    second_table_curdata = {}
    second_table_sorted_filter_cond = []
    second_table_mapfilter_cond = {}
    second_table_mapfilter_erro = {}

    for filter in sorted_filters:
        second_table_sorted_filter_cond.append(filter[0])
    for filter in second_table_sorted_filter_cond:
        second_table_mapfilter_cond[filter] = 0
        second_table_mapfilter_erro[filter] = 0
    
    remaining_second_table_sorted_filter_cond = second_table_sorted_filter_cond.copy()

    while remaining_second_table_sorted_filter_cond:
        #print("remaining_second_table_sorted_filter_cond: ",remaining_second_table_sorted_filter_cond)
        filter_cond_tochange = []
        filter_cond = remaining_second_table_sorted_filter_cond[0]
        print("filter_cond: ",filter_cond)
        second_table_mapfilter_cond[filter_cond] += 1
        if second_table_mapfilter_cond[filter_cond] >= 2:
            if filter_cond in remaining_second_table_sorted_filter_cond:
                remaining_second_table_sorted_filter_cond.remove(filter_cond)
                tochange_filter = [filter_cond,'false']
                filter_cond_tochange.append(tochange_filter)
            continue

        elif second_table_mapfilter_cond[filter_cond] == 1:
            attribute = filter_cond.split()[0].strip()
            key_sentences = ""
            for sentence in attributes[attribute]['key_sentences']:
                key_sentences += sentence
            print("key_sentence: ",key_sentences)
            
            answer = ask_completion4filtercondANDattr(str(remaining_attributes),str(remaining_second_table_sorted_filter_cond),key_sentences)
            actual_token += len(key_sentences.split())+100
            answer = remove_punctuation(answer)
            print("answer: \n", answer)
            #attributes_cur_all = answer.split("$$")[1]
            try:
                attributes_cur_all = answer.split("$$")[1]
                filter_cond_cur_all = answer.split("$$")[0]
                print("attributes_cur_all: ", attributes_cur_all)
                print("filter_cond_cur_all: ", filter_cond_cur_all)
            except:
                print("error")
                print("answer: ", answer)
                if 'second_table_output NOT IN REQUIRED FORMAT' in answer and second_table_mapfilter_erro[filter_cond] >= 1:
                    if filter_cond in remaining_second_table_sorted_filter_cond:
                        remaining_second_table_sorted_filter_cond.remove(filter_cond)
                second_table_mapfilter_cond[filter_cond] -= 1
                second_table_mapfilter_erro[filter_cond] += 1
                tochange_filter = [filter_cond,'false']
                actual_token -= len(key_sentences.split())+100
                filter_cond_tochange.append(tochange_filter)
                
                for ask_filter_cond,bool_value in filter_cond_tochange:
                        sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
                sql_copy2 = sql_copy
                print("sql_copy: ",sql_copy)
                bool_value_set_true = calculate_bool_value_true(sql_copy)
                bool_value_set_false = calculate_bool_value_false(sql_copy2)
                print("bool_value_set_true: ",bool_value_set_true)
                print("bool_value_set_false: ",bool_value_set_false)
                
                if bool_value_set_true == False:
                    skipthefile = 1
                    break
                
                if bool_value_set_false == True:
                    skipthefile = 0
                    break
                else:
                    skipthefile = 1
                continue

            
            try:
                minidatas = attributes_cur_all.split("##")
                for minidata in minidatas:
                    dataattr,datavalue = minidata.split(":")
                    second_table_curdata[dataattr] = datavalue
                    if dataattr in remaining_attributes:
                        if 'NAN' not in datavalue:
                            remaining_attributes.remove(dataattr)
                            print("remaining_attributes: ",remaining_attributes)
            except:
                print("###############")
                print("error attributes")
                print("attribute_cur_all: ", attributes_cur_all)
            
            try:
                minifilters = filter_cond_cur_all.split("##")
                for minifilter in minifilters:
                    filter_cond_cur,bool_value = minifilter.split(":")
                    tochange_filter = [filter_cond_cur,bool_value]
                    if 'NAN' not in bool_value:
                        filter_cond_tochange.append(tochange_filter)
                        print("filter_cond: ",filter_cond_cur)
                        if filter_cond_cur in remaining_second_table_sorted_filter_cond:
                            remaining_second_table_sorted_filter_cond.remove(filter_cond_cur)
                        print("remaining_second_table_sorted_filter_cond: ",remaining_second_table_sorted_filter_cond)
            except:
                print("###############")
                print("error filter_cond")
                actual_token -= len(key_sentences.split())+100
                print("filter_cond_cur_all: ", filter_cond_cur_all)
        
        
        for ask_filter_cond,bool_value in filter_cond_tochange:
            sql_copy = sql_copy.replace(ask_filter_cond,bool_value)
        sql_copy2 = sql_copy
        print("sql_copy: ",sql_copy)
        bool_value_set_true = calculate_bool_value_true(sql_copy)
        bool_value_set_false = calculate_bool_value_false(sql_copy2)
        print("bool_value_set_true: ",bool_value_set_true)
        print("bool_value_set_false: ",bool_value_set_false)
        . 
        if bool_value_set_true == False:
            skipthefile = 1
            break
        
        if bool_value_set_false == True:
            skipthefile = 0
            break
        else:
            skipthefile = 1
        print("\n")
        #remaining_second_table_sorted_filter_cond.remove(filter_cond)



    
    if skipthefile == 0:
        print("###################抽取对应的属性#########################\n")
        
        key_sentences = ""
        
        second_table_mapattr = {}
        for attr in select_attributes:
            second_table_mapattr[attr] = 0
        print("remaining_attributes: ",remaining_attributes)
        while remaining_attributes:
            #attr = random.choice(remaining_attributes)
            
            attr = remaining_attributes[0]
            second_table_mapattr[attr] += 1
            if second_table_mapattr[attr] >= 2:
                remaining_attributes.remove(attr)
                second_table_curdata[attr] = "NAN"
                continue
            if attr in attributes:
                key_sentences = ""
                for sentence in attributes[attr]['key_sentences']:
                    key_sentences += sentence + " "
                print("attribute: ",attr)
                print("key_sentences: ", key_sentences)
                answer = ask_completion4Multattribute(str(remaining_attributes), key_sentences)
                actual_token += len(key_sentences.split())+75
                answer = remove_punctuation(answer)
                print("answer: \n", answer)
                try:
                    minidatas = answer.split("##")
                    for minidata in minidatas:
                        dataattr,datavalue = minidata.split(":")
                        second_table_curdata[dataattr] = datavalue
                        if dataattr in remaining_attributes:
                            if 'NAN' not in datavalue:
                                remaining_attributes.remove(dataattr)
                except:
                    print("error extract attributes")
                    actual_token -= len(key_sentences.split())+75
                    print("answer: ", answer) 
       
             
        if len(second_table_curdata) != 0:
            second_table_curdata['file'] = onefile
            second_table_output = second_table_output.append(second_table_curdata,ignore_index=True)
            #data = data.append(second_table_curdata,ignore_index=True)
            
            second_table_output.dropna(axis=0, how='all', inplace=True) 
    
    print("actual_token: ",actual_token)
    all_actual_token += actual_token
    print("all_actual_token: ",all_actual_token)

end_time = time.time()
print("all_actual_token: ",all_actual_token)
print("all_total_token: ",all_total_token)
print("time: ",end_time-start_tie)



with open(second_infor, "w") as file:
    file.write("all_actual_token: "+str(all_actual_token)+"\n")
    file.write("all_total_token: "+str(all_total_token)+"\n")
    file.write("time: "+str(end_time-start_tie)+"\n")

second_table_output


In [ ]:

final_output = pd.merge(first_table_output, second_table_output, on=join_attr, how='inner')
final_output